#1. Read Bronze

In [0]:
from pyspark.sql import functions as F

catalog = "adb_ecommerce_de_7405607562556202"

reviews_df = spark.table(f"{catalog}.ecommerce_bronze.reviews")

#2. Clean + standardize

For Reviews, we'll keep it simple:

- Remove duplicate reviews based on the natural key.
- Trim IDs.
- Standardize review score as integer if it's already numeric, we leave it alone.
- Clean the review text fields without unnecessarily modifying the actual text.

In [0]:
silver_reviews = (
    reviews_df
    .dropDuplicates(["review_id"])
    .withColumn("review_id", F.trim(F.col("review_id")))
    .withColumn("order_id", F.trim(F.col("order_id")))
    .withColumn("review_score", F.col("review_score").cast("int"))
    .withColumn(
        "review_creation_date",
        F.to_timestamp(F.col("review_creation_date"))
    )
    .withColumn(
        "review_answer_timestamp",
        F.to_timestamp(F.col("review_answer_timestamp"))
    )
)

#Other miscellanous from previous tables

In [0]:
reviews_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in reviews_df.columns
]).show()

In [0]:
reviews_df.filter(
    F.col("order_id").isNull() |
    F.col("review_score").isNull()
).show(20, truncate=False)

In [0]:
catalog = "adb_ecommerce_de_7405607562556202"

tables = [
    "customers",
    "orders",
    "order_items",
    "payments",
    "reviews",
    "products",
    "sellers",
    "geolocation",
    "category_translation"
]

for table in tables:
    print(f"\n{'=' * 70}")
    print(f"TABLE: {table}")
    print(f"{'=' * 70}")

    df = spark.table(f"{catalog}.ecommerce_bronze.{table}")

    df.printSchema()
    df.show(3, truncate=False)

In [0]:
reviews_df.filter(
    F.col("review_score").rlike(".*-.*:.*")
).show(20, truncate=False)

In [0]:
raw_reviews = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .csv(
        "abfss://ecommerce@mayankecommerce017.dfs.core.windows.net/raw/reviews/olist_order_reviews_dataset.csv"
    )
)

raw_reviews.show(5, truncate=False)

In [0]:
bronze_reviews_path = (
    "abfss://ecommerce@mayankecommerce017.dfs.core.windows.net/"
    "bronze/reviews"
)

raw_reviews.write \
    .format("delta") \
    .mode("overwrite") \
    .save(bronze_reviews_path)

In [0]:
reviews_bronze = spark.table(
    f"{catalog}.ecommerce_bronze.reviews"
)

reviews_bronze.show(5, truncate=False)

#3. Write to ADLS

In [0]:
silver_path = (
    "abfss://ecommerce@mayankecommerce017.dfs.core.windows.net/"
    "silver/reviews"
)

silver_reviews.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)

#4.Register

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ecommerce_silver.reviews
USING DELTA
LOCATION '{silver_path}'
""")